# 03 — Build Profiles and Execution Metrics

This notebook builds historical workload profiles and point-in-time execution features from the canonical Google subset execution dataset.

For each execution, historical features are calculated using only executions from the same `recurrence_key` that have completed by the current execution's start time. This prevents unavailable future outcomes from leaking into pre-execution recommendations.

### Input
`data/processed/google_subset_v1/google_subset_executions_v1.parquet`

### Outputs
- `data/processed/google_subset_v1/job_profiles_google_subset_history_v1.parquet`
- `data/processed/google_subset_v1/job_metrics_v0.parquet`

These artifacts provide point-in-time historical workload profiles and execution-level metrics for downstream splitting, baseline generation, and recommendation evaluation.

## Imports and Paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# Find repository root
REPO_ROOT = Path.cwd()

while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

INPUT_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "google_subset_v1"
    / "google_subset_executions_v1.parquet"
)

OUTPUT_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "google_subset_v1"
)

print("Repository root:", REPO_ROOT)
print("Input:", INPUT_PATH)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Canonical execution dataset not found:\n{INPUT_PATH}\n"
        "Run 02_reconstructExecutions.ipynb first."
    )

executions = pd.read_parquet(INPUT_PATH)

print(f"Loaded {len(executions):,} executions")
print(f"Columns: {len(executions.columns)}")

executions.head()

Repository root: /Users/sofiasilva/Masters-project/masters-project
Input: /Users/sofiasilva/Masters-project/masters-project/data/processed/google_subset_v1/google_subset_executions_v1.parquet
Loaded 253,585 executions
Columns: 26


,execution_id,collection_id,instance_index,candidate_instance_key,episode_number,execution_start_time_us,execution_end_time_us,observation_count,machine_count,requested_cpu_ncu,...,runtime_sec,slack_cpu_ncu,slack_memory_normalized,user,collection_name,collection_logical_name,user_clean,logical_workload_name,recurrence_key,recurring_job_id
0,100286199||134||episode_1,100286199,134,100286199||134,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
1,100286199||250||episode_1,100286199,250,100286199||250,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
2,100286199||269||episode_1,100286199,269,100286199||269,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
3,100286199||321||episode_1,100286199,321,100286199||321,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...
4,100286199||323||episode_1,100286199,323,100286199||323,1,2292000000000,2292300000000,1,1,0.04657,...,300.0,0.035416,0.026697,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,B9j+qbUNwWym1Pfz88WG3lUG/EIH24AL3gPOzqgazH0=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=,pTO4ZKUMlGLyiuylxGJMdQYYblO1KZOFOrr2xwx7sjQ=,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...


## Complete Schema

In [2]:
executions.info()
executions.columns.tolist()
executions[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
        "runtime_sec",
        "requested_cpu_ncu",
        "requested_memory_normalized",
        "peak_cpu_ncu",
        "peak_memory_normalized",
    ]
].head(10)

<class 'pandas.DataFrame'>
RangeIndex: 253585 entries, 0 to 253584
Data columns (total 26 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   execution_id                 253585 non-null  str    
 1   collection_id                253585 non-null  int64  
 2   instance_index               253585 non-null  int64  
 3   candidate_instance_key       253585 non-null  str    
 4   episode_number               253585 non-null  int64  
 5   execution_start_time_us      253585 non-null  Int64  
 6   execution_end_time_us        253585 non-null  Int64  
 7   observation_count            253585 non-null  int64  
 8   machine_count                253585 non-null  int64  
 9   requested_cpu_ncu            252811 non-null  float64
 10  requested_memory_normalized  252811 non-null  float64
 11  average_cpu_ncu              253585 non-null  float64
 12  average_memory_normalized    253585 non-null  float64
 13  peak_cpu_n

,execution_id,recurrence_key,execution_start_time_us,runtime_sec,requested_cpu_ncu,requested_memory_normalized,peak_cpu_ncu,peak_memory_normalized
0,100286199||134||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
1,100286199||250||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
2,100286199||269||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
3,100286199||321||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
4,100286199||323||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2292000000000,300.0,0.046570,0.028015,0.011154,0.001318
5,100287787||122||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,2248500000000,300.0,0.032410,0.007294,0.010040,0.003429
6,100288420||58||episode_1,iTVkGxgmnzN6bSosM7QLqyRpmOAxn1/H+cA2UBz/20g=||...,976200000000,300.0,0.045715,0.028870,0.013321,0.000992
7,10084165805||105||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,300.0,0.070435,0.041626,0.015488,0.005989
8,10084165805||112||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,300.0,0.070435,0.041626,0.015488,0.005989
9,10084165805||17||episode_1,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=||...,958800000000,300.0,0.070435,0.041626,0.015488,0.005989


## Validate Canonical Schema

Verify that the execution dataset contains the fields required to construct historical workload profiles and point-in-time features.

In [3]:
required_columns = [
    "execution_id",
    "recurrence_key",
    "execution_start_time_us",
    "execution_end_time_us",
    "runtime_sec",
    "requested_cpu_ncu",
    "requested_memory_normalized",
    "peak_cpu_ncu",
    "peak_memory_normalized",
]

missing_columns = [
    col for col in required_columns
    if col not in executions.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required canonical columns: {missing_columns}"
    )

print("All required canonical columns are present.")

print("Execution-level integrity checks")
print("--------------------------------")

print(f"Rows:                    {len(executions):,}")
print(f"Unique execution IDs:    {executions['execution_id'].nunique():,}")
print(f"Unique recurrence keys:  {executions['recurrence_key'].nunique():,}")
print(f"Duplicate execution IDs: {executions['execution_id'].duplicated().sum():,}")
print(f"Missing recurrence keys: {executions['recurrence_key'].isna().sum():,}")
print(f"Invalid runtimes (<= 0): {(executions['runtime_sec'] <= 0).sum():,}")

All required canonical columns are present.
Execution-level integrity checks
--------------------------------
Rows:                    253,585
Unique execution IDs:    253,585
Unique recurrence keys:  2,280
Duplicate execution IDs: 0
Missing recurrence keys: 0
Invalid runtimes (<= 0): 0


## Order Executions Chronologically

Executions are grouped by `recurrence_key` and ordered by execution start time.

This ordering establishes the temporal sequence needed to construct historical features using only information available before each execution.

In [4]:
history = executions.sort_values(
    by=[
        "recurrence_key",
        "execution_start_time_us",
        "execution_id",
    ]
).reset_index(drop=True)

history[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
    ]
].head(20)

,execution_id,recurrence_key,execution_start_time_us
0,83212368472||1008||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
1,83212368472||1042||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
2,83212368472||1056||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
3,83212368472||1096||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
4,83212368472||1099||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
5,83212368472||1113||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
6,83212368472||1119||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
7,83212368472||1221||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
8,83212368472||1279||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000
9,83212368472||1332||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000


## Build Point-in-Time History

Historical features use only executions from the same `recurrence_key` that have completed by the time the current execution starts.

For an execution starting at time `t`, a previous execution is eligible for history only when:

`execution_end_time_us <= t`

This ensures that final execution outcomes such as peak CPU, peak memory, and runtime are actually available at recommendation time.

`prior_execution_count` represents the number of individual completed execution rows available before the current execution starts. It does not represent the number of distinct `recurrence_key` values or distinct start-time groups.

In [5]:
# Count executions occurring at each workload/start-time point.
time_groups = (
    history.groupby(
        ["recurrence_key", "execution_start_time_us"],
        sort=False,
    )
    .size()
    .rename("executions_at_start_time")
    .reset_index()
)

history = history.merge(
    time_groups,
    on=["recurrence_key", "execution_start_time_us"],
    how="left",
    validate="many_to_one",
)

history[
    [
        "execution_id",
        "recurrence_key",
        "execution_start_time_us",
        "execution_end_time_us",
        "executions_at_start_time",
    ]
].head(20)

,execution_id,recurrence_key,execution_start_time_us,execution_end_time_us,executions_at_start_time
0,83212368472||1008||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
1,83212368472||1042||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
2,83212368472||1056||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
3,83212368472||1096||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
4,83212368472||1099||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
5,83212368472||1113||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
6,83212368472||1119||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
7,83212368472||1221||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
8,83212368472||1279||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34
9,83212368472||1332||episode_1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,34


## Build Historical Resource Features

Historical CPU, memory, and runtime statistics are calculated for each recurring workload.

For an execution starting at time `t`, the historical profile contains only executions from the same `recurrence_key` that have completed by time `t`:

`execution_end_time_us <= t`

This ensures that peak CPU, peak memory, and runtime values are available before they are included in the historical profile.

In [6]:
time_point_summary = (
    history.groupby("recurrence_key")["execution_start_time_us"]
    .nunique()
)

print("Distinct start times per recurrence key")
print("---------------------------------------")
print(time_point_summary.describe())

print(
    "\nTotal recurrence/time groups:",
    history[
        ["recurrence_key", "execution_start_time_us"]
    ].drop_duplicates().shape[0]
)

Distinct start times per recurrence key
---------------------------------------
count    2280.000000
mean        2.139912
std         6.999918
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       274.000000
Name: execution_start_time_us, dtype: float64

Total recurrence/time groups: 4879


In [7]:
# Historical resource fields to profile
profile_fields = {
    "peak_cpu_ncu": "hist_peak_cpu",
    "peak_memory_normalized": "hist_peak_memory",
    "runtime_sec": "hist_runtime",
}

historical_profiles = []

for recurrence_key, group in history.groupby("recurrence_key", sort=False):

    group = group.sort_values(
        ["execution_start_time_us", "execution_id"]
    )

    for start_time, current_rows in group.groupby(
        "execution_start_time_us",
        sort=True,
    ):

        # Only executions completed by the current start time
        # are available as pre-execution history.
        prior_df = group[
            group["execution_end_time_us"] <= start_time
        ]

        record = {
            "recurrence_key": recurrence_key,
            "execution_start_time_us": start_time,
            "prior_execution_count": len(prior_df),
        }

        for field, prefix in profile_fields.items():

            values = prior_df[field].dropna()

            record[f"{prefix}_count"] = len(values)

            if len(values) > 0:
                record[f"{prefix}_mean"] = values.mean()
                record[f"{prefix}_std"] = values.std()
                record[f"{prefix}_p50"] = values.quantile(0.50)
                record[f"{prefix}_p90"] = values.quantile(0.90)
                record[f"{prefix}_p95"] = values.quantile(0.95)
                record[f"{prefix}_p99"] = values.quantile(0.99)

            else:
                record[f"{prefix}_mean"] = np.nan
                record[f"{prefix}_std"] = np.nan
                record[f"{prefix}_p50"] = np.nan
                record[f"{prefix}_p90"] = np.nan
                record[f"{prefix}_p95"] = np.nan
                record[f"{prefix}_p99"] = np.nan

        historical_profiles.append(record)

historical_profiles = pd.DataFrame(historical_profiles)

print(
    f"Created {len(historical_profiles):,} "
    "point-in-time historical profiles."
)

historical_profiles.head()

Created 4,879 point-in-time historical profiles.


,recurrence_key,execution_start_time_us,prior_execution_count,hist_peak_cpu_count,hist_peak_cpu_mean,hist_peak_cpu_std,hist_peak_cpu_p50,hist_peak_cpu_p90,hist_peak_cpu_p95,hist_peak_cpu_p99,...,hist_peak_memory_p90,hist_peak_memory_p95,hist_peak_memory_p99,hist_runtime_count,hist_runtime_mean,hist_runtime_std,hist_runtime_p50,hist_runtime_p90,hist_runtime_p95,hist_runtime_p99
0,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,116400000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
2,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,359400000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
3,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,45600000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
4,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,1200900000000,1,1,0.062744,NaN,0.062744,0.062744,0.062744,0.062744,...,0.005478,0.005478,0.005478,1,300.0,<NA>,300.0,300.0,300.0,300.0


In [8]:
count_checks = []

for _, row in historical_profiles.iterrows():

    expected_count = len(
        executions[
            (executions["recurrence_key"] == row["recurrence_key"])
            & (
                executions["execution_end_time_us"]
                <= row["execution_start_time_us"]
            )
        ]
    )

    count_checks.append(
        row["prior_execution_count"] == expected_count
    )

print(
    "Prior execution count mismatches:",
    len(count_checks) - sum(count_checks),
)

assert all(count_checks)

Prior execution count mismatches: 0


## Attach Historical Profiles to Executions

Merge the point-in-time historical profile features back onto each execution.

Executions sharing the same recurrence key and start time receive the same historical profile because only executions that completed by that start time are considered historical.

In [9]:
job_metrics = history.merge(
    historical_profiles,
    on=[
        "recurrence_key",
        "execution_start_time_us",
    ],
    how="left",
    validate="many_to_one",
)

print(f"Execution rows: {len(history):,}")
print(f"Job metric rows: {len(job_metrics):,}")
print(f"Columns: {len(job_metrics.columns)}")

job_metrics.head()

print("Historical profile merge validation")
print("-----------------------------------")

print(
    "Duplicate execution IDs:",
    job_metrics["execution_id"].duplicated().sum()
)

print(
    "Missing profile counts:",
    job_metrics["hist_peak_cpu_count"].isna().sum()
)

print(
    "Cold-start executions:",
    (job_metrics["prior_execution_count"] == 0).sum()
)

print(
    "Executions with history:",
    (job_metrics["prior_execution_count"] > 0).sum()
)

assert len(job_metrics) == len(history)
assert job_metrics["execution_id"].is_unique
assert job_metrics["hist_peak_cpu_count"].notna().all()

print("\nMerge validation passed.")

Execution rows: 253,585
Job metric rows: 253,585
Columns: 49
Historical profile merge validation
-----------------------------------
Duplicate execution IDs: 0
Missing profile counts: 0
Cold-start executions: 80801
Executions with history: 172784

Merge validation passed.


## Validate Point-in-Time History

Verify that historical profile features are calculated only from executions that completed by the time the current execution starts.

This check independently reconstructs the eligible completed history for selected executions and compares the expected statistics against the stored historical features.

In [10]:
# Select several execution time points that have prior history.
validation_rows = (
    job_metrics[
        job_metrics["prior_execution_count"] > 0
    ]
    .drop_duplicates(
        subset=[
            "recurrence_key",
            "execution_start_time_us",
        ]
    )
    .head(10)
)

checks = []

for _, row in validation_rows.iterrows():

    prior = executions[
    (executions["recurrence_key"] == row["recurrence_key"])
    & (
        executions["execution_end_time_us"]
        <= row["execution_start_time_us"]
        )
    ]

    checks.append({
        "recurrence_key": row["recurrence_key"],
        "execution_start_time_us": row["execution_start_time_us"],

        "stored_prior_count": row["prior_execution_count"],
        "expected_prior_count": len(prior),

        "stored_cpu_mean": row["hist_peak_cpu_mean"],
        "expected_cpu_mean": prior["peak_cpu_ncu"].mean(),

        "stored_cpu_p95": row["hist_peak_cpu_p95"],
        "expected_cpu_p95": prior["peak_cpu_ncu"].quantile(0.95),
       
        "stored_cpu_p90": row["hist_peak_cpu_p90"],
        "expected_cpu_p90": prior["peak_cpu_ncu"].quantile(0.90),

        "stored_cpu_p99": row["hist_peak_cpu_p99"],
        "expected_cpu_p99": prior["peak_cpu_ncu"].quantile(0.99),

        "stored_memory_mean": row["hist_peak_memory_mean"],
        "expected_memory_mean": prior["peak_memory_normalized"].mean(),

        "stored_memory_p95": row["hist_peak_memory_p95"],
        "expected_memory_p95": prior["peak_memory_normalized"].quantile(0.95),

        "stored_memory_p90": row["hist_peak_memory_p90"],
        "expected_memory_p90": prior["peak_memory_normalized"].quantile(0.90),

        "stored_memory_p99": row["hist_peak_memory_p99"],
        "expected_memory_p99": prior["peak_memory_normalized"].quantile(0.99),
        
        "stored_runtime_mean": row["hist_runtime_mean"],
        "expected_runtime_mean": prior["runtime_sec"].mean(),

        "stored_runtime_p95": row["hist_runtime_p95"],
        "expected_runtime_p95": prior["runtime_sec"].quantile(0.95),
        
        "stored_runtime_p90": row["hist_runtime_p90"],
        "expected_runtime_p90": prior["runtime_sec"].quantile(0.90),

        "stored_runtime_p99": row["hist_runtime_p99"],
        "expected_runtime_p99": prior["runtime_sec"].quantile(0.99),
    })

validation_check = pd.DataFrame(checks)

validation_check

count_match = (
    validation_check["stored_prior_count"]
    == validation_check["expected_prior_count"]
).all()

feature_pairs = [
    ("stored_cpu_mean", "expected_cpu_mean"),
    ("stored_cpu_p90", "expected_cpu_p90"),
    ("stored_cpu_p95", "expected_cpu_p95"),
    ("stored_cpu_p99", "expected_cpu_p99"),

    ("stored_memory_mean", "expected_memory_mean"),
    ("stored_memory_p90", "expected_memory_p90"),
    ("stored_memory_p95", "expected_memory_p95"),
    ("stored_memory_p99", "expected_memory_p99"),

    ("stored_runtime_mean", "expected_runtime_mean"),
    ("stored_runtime_p90", "expected_runtime_p90"),
    ("stored_runtime_p95", "expected_runtime_p95"),
    ("stored_runtime_p99", "expected_runtime_p99"),
]

feature_matches = {}

for stored, expected in feature_pairs:
    feature_matches[stored] = np.allclose(
        validation_check[stored],
        validation_check[expected],
        equal_nan=True,
    )

print("Prior execution counts match:", count_match)

for feature, matches in feature_matches.items():
    print(f"{feature} matches:", matches)

assert count_match
assert all(feature_matches.values())

print("\nPoint-in-time history validation passed.")

Prior execution counts match: True
stored_cpu_mean matches: True
stored_cpu_p90 matches: True
stored_cpu_p95 matches: True
stored_cpu_p99 matches: True
stored_memory_mean matches: True
stored_memory_p90 matches: True
stored_memory_p95 matches: True
stored_memory_p99 matches: True
stored_runtime_mean matches: True
stored_runtime_p90 matches: True
stored_runtime_p95 matches: True
stored_runtime_p99 matches: True

Point-in-time history validation passed.


## Validate Cold Starts and Missing Values

Cold-start executions have no prior execution history, so historical statistics should remain unavailable rather than being imputed.

Missing source measurements are also preserved. Resource-specific historical counts may therefore be smaller than the total number of prior executions.

In [11]:
cold_start = job_metrics[
    job_metrics["prior_execution_count"] == 0
]

history_available = job_metrics[
    job_metrics["prior_execution_count"] > 0
]

historical_stat_columns = [
    "hist_peak_cpu_mean",
    "hist_peak_cpu_std",
    "hist_peak_cpu_p50",
    "hist_peak_cpu_p90",
    "hist_peak_cpu_p95",
    "hist_peak_cpu_p99",

    "hist_peak_memory_mean",
    "hist_peak_memory_std",
    "hist_peak_memory_p50",
    "hist_peak_memory_p90",
    "hist_peak_memory_p95",
    "hist_peak_memory_p99",

    "hist_runtime_mean",
    "hist_runtime_std",
    "hist_runtime_p50",
    "hist_runtime_p90",
    "hist_runtime_p95",
    "hist_runtime_p99",
]

print("Cold-start validation")
print("---------------------")
print(f"Cold-start executions: {len(cold_start):,}")
print(
    "Cold starts with non-null historical stats:",
    cold_start[historical_stat_columns].notna().any(axis=1).sum()
)

print("\nExecutions with history")
print("-----------------------")
print(f"Executions with prior history: {len(history_available):,}")

print("\nHistorical feature missing values:")
print(
    history_available[
        historical_stat_columns
    ].isna().sum()
)

assert (
    cold_start[historical_stat_columns]
    .notna()
    .any(axis=1)
    .sum()
    == 0
)

assert (
    cold_start["hist_peak_cpu_count"] == 0
).all()

assert (
    cold_start["hist_peak_memory_count"] == 0
).all()

assert (
    cold_start["hist_runtime_count"] == 0
).all()

print("Cold-start validation passed.")

Cold-start validation
---------------------
Cold-start executions: 80,801
Cold starts with non-null historical stats: 0

Executions with history
-----------------------
Executions with prior history: 172,784

Historical feature missing values:
hist_peak_cpu_mean          0
hist_peak_cpu_std        1057
hist_peak_cpu_p50           0
hist_peak_cpu_p90           0
hist_peak_cpu_p95           0
hist_peak_cpu_p99           0
hist_peak_memory_mean    2357
hist_peak_memory_std     3414
hist_peak_memory_p50     2357
hist_peak_memory_p90     2357
hist_peak_memory_p95     2357
hist_peak_memory_p99     2357
hist_runtime_mean           0
hist_runtime_std         1057
hist_runtime_p50            0
hist_runtime_p90            0
hist_runtime_p95            0
hist_runtime_p99            0
dtype: int64
Cold-start validation passed.


### Target Ownership

This stage creates point-in-time historical profile features and preserves each execution's observed outcomes, including `peak_cpu_ncu` and `peak_memory_normalized`.

It does not create a shifted next-execution prediction target. Any target shifting required for downstream modeling is created in the downstream splitting and modeling stage after chronological ordering and leakage controls are established.

## Save Stage Outputs

Save the point-in-time recurrence profiles and execution-level metrics for downstream splitting, baseline generation, and recommendation evaluation.

These artifacts preserve the canonical execution schema and historical features without applying a cold-start threshold or train/validation/test split.

In [12]:
PROFILE_OUTPUT_PATH = (
    OUTPUT_DIR / "job_profiles_google_subset_history_v1.parquet"
)

METRICS_OUTPUT_PATH = (
    OUTPUT_DIR / "job_metrics_v0.parquet"
)

historical_profiles.to_parquet(
    PROFILE_OUTPUT_PATH,
    index=False,
)

job_metrics.to_parquet(
    METRICS_OUTPUT_PATH,
    index=False,
)

print("Saved stage outputs")
print("-------------------")
print(f"Profiles: {PROFILE_OUTPUT_PATH}")
print(f"Metrics:  {METRICS_OUTPUT_PATH}")

Saved stage outputs
-------------------
Profiles: /Users/sofiasilva/Masters-project/masters-project/data/processed/google_subset_v1/job_profiles_google_subset_history_v1.parquet
Metrics:  /Users/sofiasilva/Masters-project/masters-project/data/processed/google_subset_v1/job_metrics_v0.parquet


## Validate Saved Outputs

Reload the saved artifacts and verify their row counts, identifiers, and execution-level integrity.

In [13]:
saved_profiles = pd.read_parquet(PROFILE_OUTPUT_PATH)
saved_metrics = pd.read_parquet(METRICS_OUTPUT_PATH)

print("Saved artifact validation")
print("-------------------------")
print(f"Profile rows: {len(saved_profiles):,}")
print(f"Metric rows:  {len(saved_metrics):,}")
print(
    "Unique metric execution IDs:",
    f"{saved_metrics['execution_id'].nunique():,}",
)

assert len(saved_profiles) == len(historical_profiles)
assert len(saved_metrics) == len(job_metrics)

assert saved_metrics["execution_id"].is_unique

assert (
    saved_metrics["runtime_sec"] > 0
).all()

assert (
    saved_metrics["recurrence_key"].notna()
).all()

print("\nSaved artifact validation passed.")

Saved artifact validation
-------------------------
Profile rows: 4,879
Metric rows:  253,585
Unique metric execution IDs: 253,585

Saved artifact validation passed.


## Summary

This notebook:

- Loaded the canonical reconstructed execution-level dataset.
- Grouped recurring workloads using `recurrence_key`.
- Ordered executions chronologically and treated executions with the same start time as simultaneous.
- Built point-in-time CPU, memory, and runtime history using only executions completed by the current execution's start time.
- Preserved cold-start executions and missing source measurements without imputation.
- Attached historical profile features to each execution.
- Validated execution uniqueness, runtime integrity, profile coverage, and point-in-time history.
- Saved historical recurrence profiles and execution-level metrics for downstream splitting, baseline generation, and recommendation evaluation.